<a href="https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/9_aprendizaje_maquina/_assets/02_polinomios_y_capacidad.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" width="300"></a>

# Notebook 2 · Polinomios, capacidad y supuestos

Acompaña la página [Polinomios y capacidad](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/). Duración: ~30 min.

Puedes recorrerlo solo en casa, de arriba abajo. Ejecuta primero la celda de
arranque. En cada bloque, anota tu predicción antes de ejecutar «Juega»;
después mueve los controles, mira el resultado y abre «Explicación».

In [ ]:
#@title ▶ Corre esta celda primero
import os, sys, glob, base64, zlib, importlib, urllib.request
URL = os.environ.get("ML_LAB_URL",
      "https://raw.githubusercontent.com/raya-lucaria/ia_o26/main/cou"
      "rse/9_aprendizaje_maquina/_assets/ml_lab.py")
REQUIERE = "1.3"                     # la versión con que se generó este notebook
_RESPALDO = "..."

def _v(texto): return tuple(int(p) for p in texto.split("."))

def _sirve(version):                 # misma X y una Y mayor o igual
    (x, y), (x0, y0) = _v(version), _v(REQUIERE)
    return x == x0 and y >= y0

def _cargar():                       # importa ml_lab DE CERO (sin la copia en memoria)
    sys.modules.pop("ml_lab", None)
    for viejo in glob.glob("__pycache__/ml_lab.*.pyc"):
        os.remove(viejo)             # un .pyc de la copia vieja taparía la nueva
    importlib.invalidate_caches()
    import ml_lab
    return ml_lab if _sirve(ml_lab.VERSION) else None

ml = None
if os.path.exists("ml_lab.py"):      # 1) la copia local, si ya es nueva
    try: ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 2) la del repositorio
    try:
        urllib.request.urlretrieve(URL, "ml_lab.py"); ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 3) el respaldo: sin red o repo aún sin esta versión
    with open("ml_lab.py", "wb") as archivo:
        archivo.write(zlib.decompress(base64.b64decode(_RESPALDO)))
    ml = _cargar()
ml.requiere(REQUIERE)
ml.tema("claro")
from ml_lab import *
print(f"Listo: ml_lab {ml.VERSION}")

Agregar potencias permite seguir mejor los datos, pero también sus accidentes.
El grado máximo $N$ limita los polinomios disponibles; el ajuste elige sus
coeficientes $\widehat\beta$. El gorro indica que se estimaron con una muestra.

$L_{\text{ent}}$ es el error cuadrático promedio en entrenamiento y $R$ es el
riesgo: la pérdida cuadrática esperada en la población. Aquí $L$ denota error,
no un lenguaje. La página [La capacidad como restricción y Brecha de generalización](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/)
conecta la elección del grado con la diferencia entre esos errores:

$$
\min_\beta L_{\text{ent}}(\beta)\quad\text{s.a.}\quad\operatorname{grado}(f_\beta)\le N,
\qquad \mathrm{brecha}=R-L_{\text{ent}}.
$$

**Predice.** Con grado 3, ¿quién acierta mejor en el borde derecho del intervalo: Taylor centrado en el medio o el ajuste a datos? ¿Y si subes Taylor a grado 9?

Anota si esperas que gane Taylor o mínimos cuadrados, y si el error de Taylor
mejorará al subir el grado. Deja «función» en «recíproca», «centro x₀» en 1
y «datos con ruido» activado. Aquí $x_0$ es el centro de Taylor, no una
observación inicial. Está dentro del intervalo, aunque no en su punto medio.

Ejecuta con «grado N» en 3 y después muévelo a 9: el control cambia ambos
polinomios. Compara los errores impresos en el borde $x=3$.
Luego puedes elegir «seno»; allí el centro se recorta al dominio $[0,2]$,
aunque el control permita ir más lejos.

In [ ]:
#@title Juega: Taylor contra mínimos cuadrados
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 0
FUNCIONES = {"recíproca": ("inverso", (0.3, 3.0)), "seno": ("seno", (0.0, 2.0))}


def bloque_1(curva_f, N, centro, ruido):
    clave, (lo, hi) = FUNCIONES[curva_f]
    centro = min(max(centro, lo + 0.05), hi)   # el centro, dentro del dominio
    sin_ruido = {} if ruido else {"ruido": ml.normal(0, 0)}
    datos = ml.PROCESOS[clave](n=20, semilla=SEMILLA, fracciones=(1, 0, 0), **sin_ruido)
    f = lambda v: datos.f_visible(pd.DataFrame({"x": v}))
    taylor = ml.taylor(clave, centro=centro, N=N)
    mco = ml.ajustar(datos, phi=ml.polinomio(N))
    ax = ml.dibujar_ajuste({f"Taylor de grado {N} en x₀ = {centro:.1f}": taylor,
                            f"mínimos cuadrados, grado {N}": mco}, datos,
                           ventana=(lo, hi))
    x = np.linspace(lo, hi, 300)
    ax.plot(x, f(x), color="green", lw=3, zorder=1, label="f verdadera")
    if clave == "inverso" and centro == 1.0:
        ax.axvline(2, ls=":", color="black", label="x = 2: Taylor deja de converger")
    ax.set(xlim=(lo, hi), ylim=(min(f(x)) - 2, max(f(x)) + 2), ylabel="y")
    ax.legend(loc="upper right", fontsize=8), plt.show()
    b = np.array([hi])
    e_t = abs(taylor(b) - f(b))[0]
    e_m = abs(mco.predecir(pd.DataFrame({"x": b})) - f(b))[0]
    print(f"en el borde x = {hi}: error de Taylor = {e_t:.3f}"
          f"   ·   de mínimos cuadrados = {e_m:.3f}")


ml.laboratorio(bloque_1, curva_f=ml.selector("función", list(FUNCIONES)),
               N=ml.deslizador("grado N", 1, 9, 1, 3),
               centro=ml.deslizador("centro x₀", 0.3, 3.0, 0.1, 1.0),
               ruido=ml.interruptor("datos con ruido", True))

**Mira.** Los puntos son los datos; la curva verde gruesa es $f$, y la leyenda distingue Taylor de mínimos cuadrados.
La franja gris queda fuera del rango de los datos; la línea vertical punteada marca $x=2$ para la recíproca con $x_0=1$.
Compara Taylor cerca del centro y en el borde derecho; si sale del panel, lee su error debajo de la gráfica.

<details><summary>Explicación</summary>

Gana mínimos cuadrados con los valores indicados: en $x=3$, el error de
Taylor es 26.667 y el del ajuste es 0.948. Al subir el grado a 9, el error
de Taylor llega aproximadamente a 1707. Más grado empeoró esa aproximación.

Taylor usa el valor y las derivadas de la función en su centro. Mínimos
cuadrados reparte el error entre los datos al minimizar $L_{\text{ent}}$.
Cuando solo tenemos observaciones, no contamos con las derivadas exactas de $f$.

La observación [El polinomio de Taylor](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/)
explica el límite: para $1/x$ centrada en $x_0=1$, la serie converge solo
en $(0,2)$. A la derecha de $x=2$, subir el grado no resuelve el problema.
Taylor es una aproximación local; su comportamiento aquí no dice que todo
polinomio de grado alto deba fallar.

</details>

**Predice.** Si subes $N$ de 4 a 12 con $n=20$, ¿$L_{\text{ent}}$ baja o sube? ¿Y $R$? ¿Y si pones $n=200$?

Anota por separado si cada error bajará, subirá o quedará casi igual.
Mantén «ruido σ_ε» en 0.5. Con «datos n» en 20, mueve «grado N» de 4 a 12;
después repite con «datos n» en 200. Compara los valores impresos y las
curvas del panel derecho. Aquí $n$ es el número de datos de entrenamiento.

In [ ]:
#@title Juega: la curva de capacidad en vivo
SEMILLA = 0
CURVAS = {}   # (n, σ) -> (datos, tabla): se calcula una vez por combinación


def curva(n, sigma):
    if (n, sigma) not in CURVAS:
        datos = ml.PROCESOS["inverso"](n=n, semilla=SEMILLA, fracciones=(1, 0, 0),
                                       ruido=ml.normal(0, sigma))
        grados = {N: (lambda df, N=N: ml.ajustar(df, phi=ml.polinomio(N)))
                  for N in range(1, 16)}
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            tabla = ml.curva_capacidad(datos, grados, conjuntos=("ent", "realidad"))
        CURVAS[(n, sigma)] = (datos, tabla)
    return CURVAS[(n, sigma)]


def bloque_2(N, n, sigma):
    datos, tabla = curva(n, sigma)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    m = ml.ajustar(datos, phi=ml.polinomio(N))
    ml.dibujar_ajuste({f"grado {N}": m}, datos, ax=a1)
    a1.set(ylim=(0, 20), ylabel="y")
    media = ml.benchmarks(datos, datos.realidad).loc["media", "cuadratica"]
    ml.dibujar_curva_capacidad(tabla, brecha=True, zonas=True, optimo=True,
                               piso=ml.piso(datos), benchmark=media, marcar=N, ax=a2)
    a2.set_yscale("log")   # R del grado 15 puede valer miles: en escala log se ve todo
    plt.show()
    L = tabla[tabla.candidato == N].set_index("conjunto")["L"]
    print(f"grado {N}: L_ent = {L['ent']:.3f}   ·   R = {L['realidad']:.3f}"
          f"   ·   brecha = {L['realidad'] - L['ent']:.3f}")
    print(f"piso = {ml.piso(datos):.3f}   ·   la media (benchmark), R = {media:.3f}")


ml.laboratorio(bloque_2, N=ml.deslizador("grado N", 1, 15, 1, 4),
               n=ml.deslizador("datos n", 20, 200, 10, 20),
               sigma=ml.deslizador("ruido σ_ε", 0.1, 1.5, 0.1, 0.5))

**Mira.** Izquierda: los puntos son datos y las curvas, $f$ y el ajuste. Derecha: círculos para $L_{\text{ent}}$ y discontinua para $R$, aproximado con observaciones nuevas.
El eje vertical es logarítmico; la brecha está sombreada. La punteada marca el piso; la raya-punto, la media como punto de comparación (*benchmark*).
La estrella señala el menor $R$ medido; el círculo destacado, el grado actual.

<details><summary>Explicación</summary>

$L_{\text{ent}}$ no sube al pasar de grado 4 a 12. Con 20 datos, $R$ aumenta;
con 200, la curva queda casi plana desde el grado 5. Con los controles
iniciales, el grado 4 da $L_{\text{ent}}=0.184$, $R=0.444$ y brecha 0.260.
El menor $R$ medido aparece en grado 6; en grado 15 llega aproximadamente a 1537.

La monotonía del mínimo de entrenamiento sí es una ley para familias
anidadas: el polinomio anterior sigue disponible al permitir un grado mayor.
La U de $R$ es de este experimento, no una ley. Aquí los grados altos
incurren en sobreajuste (*overfitting*): mejoran en entrenamiento y empeoran
al recibir observaciones nuevas. Con más datos o poco ruido, la subida puede
ser mucho menos marcada.

La definición [Brecha de generalización y el teorema El entrenamiento miente a tu favor, en promedio](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/)
explican $\mathrm{brecha}=R-L_{\text{ent}}$. Bajo los supuestos del teorema,
su promedio sobre muestras no es negativo; una muestra particular puede
tener brecha negativa. Tampoco hay una ley que obligue a la brecha a crecer
con cada aumento de grado.

El control mueve la desviación estándar del ruido, $\sigma_\varepsilon$.
Su cuadrado, $\sigma_\varepsilon^2$, mide la dispersión cuadrática del ruido
alrededor de su media cero; aquí determina el piso, 0.250 con los valores
iniciales. La estrella de la gráfica marca una comparación entre ajustes
de esta muestra, no un óptimo teórico al que debamos poner estrella en la notación.

</details>

**Predice.** En 200 muestras, ¿cuántas veces elegirá $L_{\text{ent}}$ el grado máximo?

Elige entre ninguna, algunas o todas. Ejecuta con «N_max» en 15 y lee el
conteo bajo los histogramas. Después mueve «N_max» hacia 5 y comprueba si
las elecciones siguen al máximo permitido. Cada muestra tiene 20 datos.

In [ ]:
#@title Juega: ¿quién elige N?
SEMILLA, REPETICIONES = 0, 200
MALLA = np.linspace(0.3, 3, 1000)   # R = E_x[(f − f̂)²] + σ², con x uniforme
grande = ml.PROCESOS["inverso"](n=20 * REPETICIONES, semilla=SEMILLA,
                                fracciones=(1, 0, 0), con_realidad=False).ent


def ajuste(x, y, N):
    """El ajuste de ml.ajustar(phi=ml.polinomio(N)), en forma cerrada (más rápido)."""
    z = lambda v: 2 * (v - x.min()) / (x.max() - x.min()) - 1
    beta = np.linalg.lstsq(np.vander(z(x), N + 1, increasing=True), y, rcond=None)[0]
    return lambda v: np.vander(z(v), N + 1, increasing=True) @ beta


def experimento(s):   # la muestra s (20 datos): L_ent y R de cada grado 1..15
    trozo = grande.iloc[20 * (s - SEMILLA):20 * (s - SEMILLA + 1)]
    x, y, filas = trozo["x"].to_numpy(), trozo["y"].to_numpy(), {}
    for N in range(1, 16):
        g = ajuste(x, y, N)
        filas[f"L{N}"] = np.mean((y - g(x)) ** 2)
        filas[f"R{N}"] = np.mean((2 + 5 / MALLA - g(MALLA)) ** 2) + 0.25
    return filas


TODAS = ml.repetir(experimento, repeticiones=REPETICIONES, semilla=SEMILLA)


def bloque_3(N_max):
    elegido = lambda c: TODAS[[f"{c}{N}" for N in range(1, N_max + 1)]].to_numpy()
    tabla = pd.DataFrame({"N_ent": elegido("L").argmin(axis=1) + 1,
                          "N_R": elegido("R").argmin(axis=1) + 1})
    ml.dibujar_elecciones(tabla)
    plt.show()
    print(f"de {REPETICIONES} muestras, L_ent eligió N = {N_max} (el máximo) "
          f"{(tabla.N_ent == N_max).sum()} veces")
    print("el N de menor R:", tabla.N_R.value_counts().sort_index().to_dict())


ml.laboratorio(bloque_3, N_max=ml.deslizador("N_max", 5, 15, 1, 15))

**Mira.** En el histograma «N_ent», cada barra cuenta las muestras que eligieron ese grado por su error de entrenamiento.
En «N_R», cada barra cuenta las que tuvieron su menor riesgo medido en ese grado; compara la concentración de ambos histogramas.
Tú eliges $N$; el ajuste elige $\widehat\beta$.

<details><summary>Explicación</summary>

Con «N_max» en 15, entrenamiento elige grado 15 en 200 de 200 muestras.
El histograma de menor $R$ se reparte entre varios grados. Volver a medir
en los mismos datos premia la libertad de ajuste, aunque no beneficie las
predicciones para observaciones nuevas.

Por familias anidadas, el grado máximo siempre alcanza un mínimo de
$L_{\text{ent}}$ entre los candidatos, salvo diferencias de aritmética.
Si hay empates, un grado menor también puede minimizarlo; la monotonía no
obliga a escoger el máximo entre candidatos empatados.

La definición [Parámetro contra hiperparámetro](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/)
distingue los coeficientes $\beta$, que se ajustan dentro de una familia,
del hiperparámetro $N$, que decide qué familia permites. Minimizar
$L_{\text{ent}}$ dentro de cada familia no resuelve cómo elegir entre ellas.

</details>

**Predice.** ¿Qué baja más $R$: el polinomio de grado 40 o la $\phi$ de dos columnas $(1,1/x)$?

Elige polinomio, recíproca o empate. Conserva «escalar x» activado, deja
«φ» en «polinomio» y lleva «grado N» a 40. Anota $R$ y cambia «φ» a
«(1, 1/x)» para compararlo. Después desactiva «escalar x» con el polinomio
y observa el aviso y los coeficientes.

$\phi$ transforma cada observación en las columnas que usa el modelo;
$\Phi$ es la matriz que las reúne para todos los datos. Incluye la columna
constante 1. Si escribimos $\phi_N$, su subíndice numérico indica solo el grado.

In [ ]:
#@title Juega: elegir φ contra fuerza bruta
SEMILLA = 0
CASO_3 = ml.caso(3, n=60, semilla=SEMILLA)


def bloque_4(phi, N, escalar):
    datos = CASO_3
    with warnings.catch_warnings():
        warnings.simplefilter("always")   # el aviso de mal condicionamiento se muestra
        if phi == "polinomio":
            m = ml.ajustar(datos, phi=ml.polinomio(N, escalar=escalar))
        else:
            m = ml.ajustar(datos, phi=ml.columnas("1/x"))
    nombre = f"polinomio de grado {N}" if phi == "polinomio" else "φ = (1, 1/x)"
    ml.dibujar_ajuste({nombre: m}, datos).set(ylim=(0, 40), ylabel="y")
    plt.show()
    ml.imprimir_Phi(m, datos.ent, filas=4)
    media = ml.benchmarks(datos, datos.despliegue).loc["media", "cuadratica"]
    print(f"columnas de Φ: {len(m.beta_gorro)}   ·   L_ent = {ml.L(m, datos.ent):.3f}"
          f"   ·   R (despliegue) = {ml.riesgo(m, datos):.3f}")
    print(f"piso = {ml.piso(datos):.3f}   ·   la media (benchmark), R = {media:.3f}")
    print(f"el mayor coeficiente en valor absoluto: {m.beta_gorro.abs().max():.3g}")


ml.laboratorio(bloque_4, phi=ml.selector("φ", ["polinomio", "(1, 1/x)"]),
               N=ml.deslizador("grado N", 1, 40, 1, 10),
               escalar=ml.interruptor("escalar x", True))

**Mira.** Los puntos son observaciones de la recíproca cerca de cero; la curva muestra el ajuste elegido.
Debajo aparecen cuatro filas de $\Phi$: compara sus dos columnas para $(1,1/x)$ con las 41 del grado 40.
Lee $R$, el piso y el benchmark; sin escalar, vigila también el aviso y el mayor coeficiente en valor absoluto.

<details><summary>Explicación</summary>

Gana $(1,1/x)$ en esta comparación. Con dos columnas alcanza $R=0.250$,
cerca del piso, y estima $\widehat\beta=(2.03,\ 5.00)$.
El polinomio inicial de grado 10 usa 11 columnas y da $R=1.279$.
La representación recíproca puede expresar la forma del proceso con menos
coeficientes: reduce tanto la capacidad como el error de aproximación.

Como explica [Elegir modelo es elegir supuestos](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/),
la ventaja depende de acertar con la forma del proceso. Elegir columnas
informativas puede ayudar más que agregar potencias. El ajuste sigue siendo
lineal en sus coeficientes, aunque aparezca $1/x$ como columna.

Sin escalar, las potencias pueden producir mal condicionamiento: la
aritmética pierde precisión y los coeficientes se vuelven enormes. Si
$L_{\text{ent}}$ sube, no se refuta la monotonía del mínimo exacto entre
familias anidadas; el cálculo ya no está alcanzándolo con precisión.
El caso 4 de la galería permite explorar ese fallo.

</details>

**Predice.** Con el seno y grado 7, ¿el error fuera de la zona de datos será mayor o menor que adentro?

Anota mayor, menor o parecido. Cambia «proceso» de «entregas» a «seno»,
deja «grado N» en 7 y «zona de uso (× el rango)» en 1.5. Compara
«dentro» y «fuera» en `R_zona`. Mueve después la zona de uso para ver
hasta dónde estás pidiendo predicciones.

Para explorar «recursivo», baja «grado N» a 1 y compara con «agregar la
columna del tiempo anterior» apagado y encendido. Ese interruptor solo
cuenta en el proceso recursivo; allí la comparación es con los pasos futuros.

In [ ]:
#@title Juega: extrapolar
SEMILLA = 0
RANGOS = {"entregas": ("entregas", 0, 6), "seno": ("seno", 0, 2), "recursivo": (
    "recursivo", 0, 0), "mezcla de logaritmos": ("mezcla_logs", 1, 5)}


def bloque_5(proceso, N, zona, anterior):
    clave, lo, hi = RANGOS[proceso]
    if clave == "recursivo":   # la x crece con el tiempo: «fuera» es el futuro
        datos = ml.PROCESOS["recursivo"](semilla=SEMILLA)
        phi = ml.polinomio(N, "x")   # y, si lo pides, el tiempo anterior observado
        phi = ml.combinar(phi, ml.columnas("y_anterior")) if anterior else phi
        m = ml.ajustar(datos, phi=phi)
        todo = ml.PROCESOS["recursivo"](n=79, futuro=1, semilla=SEMILLA,   # la misma
                                        con_realidad=False).ent      # serie, entera
        ax = plt.subplots(figsize=(8, 3.8))[1]
        ax.scatter(todo.t, todo.y, s=16, c=np.where(todo.t < 60, "C0", "gray"),
                   label="la serie (en gris, lo que pasó después)")
        ax.plot(todo.t, m.predecir(todo), color="C1", label="predicción a un paso")
        ax.set(xlabel="paso (x crece con el tiempo)", ylabel="y"), ax.legend()
        plt.show()
        print(f"dentro: L = {ml.L(m, datos.realidad):.3f}  ·  a un paso en el "
              f"futuro: R = {ml.riesgo(m, datos):.3f}  ·  piso = {ml.piso(datos):.3f}")
        return
    datos = ml.PROCESOS[clave](semilla=SEMILLA, fracciones=(1, 0, 0))
    hasta, m = lo + zona * (hi - lo), ml.ajustar(datos, phi=ml.polinomio(N))
    f = lambda v: datos.f_visible(pd.DataFrame({"x": v}))
    x = np.linspace(lo, hasta, 400)
    ax = ml.dibujar_ajuste({f"grado {N}": m}, datos, ventana=(lo, hasta))
    ax.plot(x, f(x), color="green", lw=3, zorder=1, label="f verdadera")
    ax.set(ylim=(min(f(x)) - 3, max(f(x)) + 3), ylabel="y"), ax.legend(), plt.show()
    if hasta > hi:
        tabla = ml.error_por_zona(m, datos, {"dentro": (lo, hi), "fuera": (hi, hasta)})
        tabla = tabla[["zona", "desde", "hasta", "R_zona"]].round(3)
        print(tabla.to_string(index=False))

ml.laboratorio(bloque_5, proceso=ml.selector("proceso", list(RANGOS)),
               N=ml.deslizador("grado N", 1, 10, 1, 7),
               zona=ml.deslizador("zona de uso (× el rango)", 1.0, 2.0, 0.1, 1.5),
               anterior=ml.interruptor("agregar la columna del tiempo anterior", False))

**Mira.** Con el seno, los puntos son datos; compara el polinomio con la curva verde de $f$ y lee `R_zona` para la franja gris exterior.
Con «recursivo», el eje horizontal ordena los pasos: los puntos distinguen entrenamiento y después; la línea es la predicción a un paso.
Compara los errores impresos dentro y en el futuro al agregar el valor anterior observado, manteniendo grado 1.

<details><summary>Explicación</summary>

El error del seno es mayor fuera: con los controles indicados, pasa de
0.097 dentro a 609 fuera. El intervalo de uso se extiende de 2 a 3;
el ajuste no recibió datos en esa extensión. En «entregas», con grado 7,
pasa de 1.824 dentro a más de un millón fuera al extender de 6 a 9 km.

La observación [Interpolar, extrapolar y el riesgo](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/)
explica por qué un buen ajuste dentro del intervalo no garantiza buenas
predicciones fuera. Allí los datos no restringen directamente al polinomio.
La tabla separa errores por zona; ampliar la ventana no cambia por sí solo
la población de despliegue que define $R$.

El recursivo muestra además una variable ausente: el tiempo de entrega
anterior observado. Con grado 1, sin esa columna, los errores son
aproximadamente 5.5 dentro y 36.4 en el futuro. Al agregarla quedan cerca
de 1.05 en ambos, frente al piso 1. La media condicional con lo visible
promedia lo que no observas; no la identificamos automáticamente con $f$.

Estas predicciones son **a un paso**: reciben el valor anterior observado,
no una predicción anterior encadenada. A varios pasos el error puede
acumularse. Con grado 7, aun agregando esa columna, la parte polinómica en
$x$ vuelve a extrapolar mal. La separación de series se estudia en la clase 5.

</details>

**Predice.** Promediando sobre las 256 funciones, ¿qué regla acierta más fuera de la muestra: el vecino más cercano o “siempre 1”?

Elige vecino, siempre 1 o empate. Ejecuta «Juega» y cambia «regla» entre
«vecino más cercano» y «siempre 1»; compara el promedio exacto impreso.
Después selecciona «mayoría (empate → 1)». Ejecuta también la siguiente
celda, «Mira: cada modelo gana en su proceso», para obtener la parte B.

In [ ]:
#@title Juega: no free lunch
from fractions import Fraction

REGLAS = {"vecino más cercano": "vecino", "mayoría (empate → 1)": "mayoria",
          "siempre 1": "siempre_1"}
TABLA_NFL = ml.no_free_lunch(entrenamiento=(0, 3, 5, 6))   # 000, 011, 101, 110


def bloque_6(regla):
    trozo = TABLA_NFL[TABLA_NFL.regla == REGLAS[regla]]
    ml.dibujar_no_free_lunch(trozo)
    plt.show()
    promedio = sum(trozo.aciertos_fuera, Fraction(0)) / len(trozo)
    print(f"{regla}: promedio exacto sobre las {len(trozo)} funciones = {promedio}")
    medias = trozo.aciertos_fuera.astype(float).groupby(trozo.rugosidad).mean()
    print("aciertos medios por rugosidad:", medias.round(3).to_dict())


ml.laboratorio(bloque_6, regla=ml.selector("regla", list(REGLAS)))

In [ ]:
#@title Mira: cada modelo gana en su proceso
SEMILLA = 0
CURVAS_B = {"suave": lambda d: 10 + 4 * d.x - 0.3 * d.x ** 2,
            "periódico": lambda d: 4 + 2 * np.sin(5 * d.x)}
RUIDOS = {"suave": 1.0, "periódico": 0.3}
filas = []
for nombre, f in CURVAS_B.items():
    datos = ml.proceso(n=60, x={"x": ml.uniforme(0, 6)}, f=f, semilla=SEMILLA,
                       ruido=ml.normal(0, RUIDOS[nombre]), fracciones=(1, 0, 0))
    polinomio = ml.ajustar(datos, phi=ml.polinomio(3))
    vecinos = ml.ajustar_otro(datos, familia="vecinos", n_vecinos=3)
    filas.append({"proceso": nombre, "polinomio N = 3": ml.riesgo(polinomio, datos),
                  "vecinos (3)": ml.riesgo(vecinos, datos)})
print("R (despliegue) de cada modelo en cada proceso:")
print(pd.DataFrame(filas).round(3).to_string(index=False))

**Mira.** En A, cada punto de la curva resume los aciertos fuera de la muestra para un grupo de funciones de igual rugosidad.
La referencia horizontal marca $1/2$; compara la curva por grupos con el promedio exacto impreso para todas las funciones.
En B, cada fila de la tabla es un proceso y cada columna, un modelo: busca el menor $R$ dentro de cada fila.

<details><summary>Explicación</summary>

Empatan exactamente: cada regla promedia $1/2$ de aciertos fuera de la
muestra sobre las 256 funciones. «Siempre 1» queda en 0.5 en cada grupo.
El vecino pasa de 1.0 en rugosidad 0 a 0.0 en rugosidad 12;
la mayoría también favorece lo suave y falla en lo rugoso.

El promedio global pondera cada grupo por su número de funciones.
No basta promediar las alturas de la curva dando el mismo peso a todos
los grupos. Al considerar todas las funciones por igual, la ventaja en
unas se compensa con la desventaja en otras. Es el alcance del teorema
[No free lunch (Wolpert, 1996)](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/polinomios-y-capacidad/).

En B, el polinomio de grado 3 gana en el proceso suave: $R=1.046$ frente
a 1.486 de vecinos. En el periódico ganan los vecinos: 0.314 frente a
2.027 del polinomio. Esta parte ilustra especialización y **no demuestra
el teorema**. Tampoco son aciertos: B compara riesgos con pérdida cuadrática.
Elegir un modelo supone apostar por ciertas formas del proceso.

</details>

**Taller.** Ejecuta el ejemplo activo: usa 20 datos de la recíproca y
`ml.taller(datos, clase=1, phi=ml.polinomio(4), N_max=15)`.
Mira la nube con $f$, la curva de capacidad y la matriz $\Phi$ de la
representación elegida, junto con sus errores y su piso.

Para comparar con $(1,1/x)$, comenta la llamada activa y descomenta la
línea con `phi=ml.columnas("1/x")`. Para el seno, descomenta todas las
líneas de su definición de `datos` y la llamada con `ml.periodicas`.
La constante 1 se incluye también en esa representación.

El último ejemplo permite cambiar la expresión de una $f$ propia.
Descomenta su definición completa y coloca después la llamada a `ml.taller`
que quieras usar. Se usa la última asignación a `datos` anterior a la
llamada. Cambia una representación a la vez y compara entrenamiento,
riesgo, piso y benchmark.

In [ ]:
datos = ml.proceso(n=20, x={"x": ml.uniforme(0.3, 3)}, f=lambda d: 2 + 5 / d.x,
                   ruido=ml.normal(0, 0.5), fracciones=(1, 0, 0))
ml.taller(datos, clase=1, phi=ml.polinomio(4), N_max=15)
# ml.taller(datos, clase=1, phi=ml.columnas("1/x"), N_max=15)   # la columna correcta
# datos = ml.proceso(n=50, x={"x": ml.uniforme(0, 2)},
#                    f=lambda d: 4 + 2 * np.sin(5 * d.x),
#                    ruido=ml.normal(0, 0.3), fracciones=(1, 0, 0))
# ml.taller(datos, clase=1, phi=ml.periodicas("x", periodo=2 * np.pi / 5), N_max=15)
# datos = ml.proceso(n=40, x={"x": ml.uniforme(-2, 2)}, f=lambda d: d.x ** 3 - d.x,
#                    ruido=ml.normal(0, 0.5), fracciones=(1, 0, 0))   # tu f

**Galería.** Pulsa los botones de los casos 3, 4, 5 y 23. En cada uno,
lee la pregunta y anota una predicción antes de abrir «¿Qué se ve?».
Usa la gráfica y los valores que aparezcan para justificar tu respuesta;
después compárala con la explicación del botón.

In [ ]:
#@title Galería: casos que se rompen a la vista
ml.galeria(clase=1, casos=[3, 4, 5, 23])

**Ejercicio.** Si más potencias bajan $L_{\text{ent}}$, ¿por qué no poner mil?

Responde usando la curva de capacidad del bloque 2, el interruptor
«escalar x» del bloque 4 y el caso 23 de la galería. Distingue el
problema de predecir observaciones nuevas del problema de calcular el ajuste.

**Pista:** ¿qué garantiza un error pequeño en los datos usados para ajustar,
y qué puedes concluir si el cálculo pierde precisión?

<details><summary>Explicación</summary>

Bajar $L_{\text{ent}}$ no garantiza bajar $R$. En el bloque 2, el grado 15
tiene un riesgo cercano a 1537: la brecha crece aunque mejore entrenamiento.
Ese crecimiento pertenece al experimento; no es una ley para toda muestra.

Además, con potencias sin escalar, el mal condicionamiento deteriora la
precisión del cálculo. Por eso un aumento observado de $L_{\text{ent}}$
puede ser un problema de aritmética. En el caso 23, con más coeficientes
que datos, el ajuste interpola: reproduce los datos de entrenamiento.
Eso tampoco asegura acertar en observaciones nuevas. Agregar potencias
no resuelve por sí solo ninguno de estos problemas.

</details>

**Ejercicio.** Deja que $L_{\text{ent}}$ elija $N$ en el bloque 3.
Mueve «N_max» de 15 a 5, anota qué grado elige y explica por qué
su elección sigue al máximo permitido. Distingue lo que eliges tú de
lo que decide el ajuste.

**Pista:** ¿deja de estar disponible el mejor polinomio anterior cuando
permites un grado mayor?

<details><summary>Explicación</summary>

Entrenamiento elige el máximo permitido en este experimento. Las familias
son anidadas: permitir otro grado conserva los polinomios anteriores y
agrega posibilidades. Por eso el mínimo de $L_{\text{ent}}$ no sube;
si hubiera empates, el máximo no tendría que ser el único minimizador.

Tú fijas el hiperparámetro $N$ y el ajuste estima $\widehat\beta$.
Para elegir el grado con datos reales usaremos datos que no tocaron el
ajuste; la clase 2 presenta cómo hacerlo. Aquí podemos comparar con $R$
porque conocemos el proceso que produjo los datos.

</details>

Te llevas: aproximar no es aprender; la capacidad se elige; los supuestos deciden.

En este laboratorio podíamos mirar detrás del escenario. Con datos reales no podemos. ¿Cómo medimos el error que importa?